# Recollida de dades

Aquest notebook genera la mostra inicial del treball: 1000 colors RGB aleatoris, el seu chroma i les imatges PNG que despres passarem als models.

La logica important esta a `scripts/utils.py`. Aqui intento deixar nomes la configuracio i les crides principals, perque si no el notebook es fa impossible de seguir.

## Configuracio

Fixem els parametres de l'experiment. La seed es important per poder regenerar exactament la mateixa mostra en un altre ordinador.

In [1]:
from pathlib import Path
import sys

N_COLORS = 1000
SEED = 23
IMAGE_SIZE = 100
NEAR_DISTANCE = 30

# Canvia a True nomes si vols tornar a generar la mostra des de zero.
REGENERATE_SAMPLE = False

# Casella de seguretat: si es posa a True, esborra CSVs, imatges PNG i logs generats.
DELETE_EXISTING_SAMPLE = False

base_dir = Path("recollida-dades") if Path("recollida-dades").exists() else Path(".")
csv_dir = base_dir / "csv"
images_dir = base_dir / "images"
logs_dir = base_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)


Carpeta base: .


## Carrega de scripts

Importem les funcions del script. Si alguna no existeix, parem aqui, perque probablement el notebook no esta trobant be `scripts/utils.py`.

In [2]:
import pandas as pd
from IPython.display import Image as DisplayImage, display

from utils import (
    build_final_sample,
    collect_model_outputs,
    delete_sample_outputs,
    generate_images_from_sample,
    generate_unique_colors,
    save_csv,
    save_rgb_sample_map,
    write_log,
)

required_functions = [
    "generate_unique_colors",
    "generate_images_from_sample",
    "save_rgb_sample_map",
    "delete_sample_outputs",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no cargados correctamente: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers esborrats: {len(removed)}")

write_log("Notebook de recollida inicialitzat", logs_dir / "pipeline.log")


'2026-04-25 01:04:14 Notebook de recollida inicialitzat'

## Generacio o carrega de la mostra de colors

Si `input_image_sample.csv` ja existeix, el carreguem i no el tornem a generar. Aixi evitem canviar la mostra sense voler.

Per regenerar-ho tot, cal posar `DELETE_EXISTING_SAMPLE = True` i `REGENERATE_SAMPLE = True` a la configuracio. Ho deixo aixi una mica explicit per no liar-la per accident.


In [3]:
input_path = csv_dir / "input_image_sample.csv"

if input_path.exists() and not REGENERATE_SAMPLE:
    input_sample = pd.read_csv(input_path)
    print(f"Mostra carregada: {input_path}")
else:
    input_sample = generate_unique_colors(n=N_COLORS, seed=SEED)
    save_csv(input_sample, input_path)
    write_log(f"Mostra generada: {input_path}", logs_dir / "pipeline.log")
    print(f"Mostra generada: {input_path}")

input_sample.head()


,image_name,hex,r,g,b,chroma
0,942A08.png,942A08,148,42,8,60.888058
1,9DD8C2.png,9DD8C2,157,216,194,23.896847
2,B74262.png,B74262,183,66,98,50.352746
3,88E307.png,88E307,136,227,7,96.314540
4,71E80C.png,71E80C,113,232,12,101.330068


## Mapa de color i comprovacio rapida de biaix

Guardem un mapa visual amb les 1000 mostres RGB. No es una prova estadistica, pero va be per veure rapidament si la mostra cobreix colors variats o si ha passat alguna cosa rara.


In [4]:
color_map_path = images_dir / "rgb_sample_map.png"

if color_map_path.exists() and not REGENERATE_SAMPLE:
    print(f"Mapa carregat: {color_map_path}")
else:
    save_rgb_sample_map(input_sample, color_map_path)
    write_log(f"Mapa RGB generat: {color_map_path}", logs_dir / "pipeline.log")
    print(f"Mapa generat: {color_map_path}")

display(DisplayImage(filename=str(color_map_path)))
input_sample[["r", "g", "b", "chroma"]].describe()


,r,g,b,chroma
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,121.618000,128.883000,127.076000,57.821372
std,73.846282,74.174086,75.510079,27.501296
min,0.000000,0.000000,0.000000,2.204753
25%,58.750000,64.000000,59.000000,36.230498
50%,114.000000,130.000000,127.000000,56.069794
75%,182.250000,193.000000,192.000000,78.992284
max,255.000000,255.000000,255.000000,125.276866


## Generacio de les imatges

Les imatges individuals tambe es reutilitzen si ja existeixen. Si falten o si activem `REGENERATE_SAMPLE`, es tornen a crear.

Cada imatge es de 100x100 pixels: 60% color objectiu, 20% color proper i 20% color aleatori. Les guardem en PNG per no modificar el color amb compressio JPEG.


In [5]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png") if path.name != "rgb_sample_map.png"}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_SAMPLE:
    image_paths = generate_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        near_distance=NEAR_DISTANCE,
        seed=SEED,
    )
    write_log(f"Imatges generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    image_paths = [path for path in image_paths if path.name != "rgb_sample_map.png"]
    print(f"Imatges ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]


(1000,
 [WindowsPath('images/942A08.png'),
  WindowsPath('images/9DD8C2.png'),
  WindowsPath('images/B74262.png')])

## Consulta als models

Aquesta part queda preparada pero no l'executo automaticament. Si la llancem sense voler, gastarem crides d'API.

Quan tinguem la clau configurada, la idea es generar `outputmodel_image_sample.csv` amb una fila per imatge i model.

In [ ]:
# from openai import OpenAI
#
# client = OpenAI()
# models = ["gpt-4o", "gpt-4o-mini"]
# model_outputs = collect_model_outputs(
#     client=client,
#     image_paths=image_paths,
#     models=models,
#     temperature=0.2,
# )
#
# output_path = csv_dir / "outputmodel_image_sample.csv"
# save_csv(model_outputs, output_path)
# write_log(f"Resultats de models guardats: {output_path}", logs_dir / "pipeline.log")

## Dataset final per analisi

Quan existeixi el CSV de sortida dels models, unim l'entrada i la sortida i calculem l'error cromatic en RGB. Aquest fitxer sera el que carregarem a R.

In [ ]:
# output_path = csv_dir / "outputmodel_image_sample.csv"
# final_path = csv_dir / "sample-colors.csv"
#
# if output_path.exists():
#     model_outputs = pd.read_csv(output_path)
#     final_sample = build_final_sample(input_sample, model_outputs)
#     save_csv(final_sample, final_path)
#     final_sample.head()
# else:
#     print("Encara no existeix outputmodel_image_sample.csv")

## Resultats generats

Comprovem que tenim el CSV d'entrada i les imatges. Si aqui surt 1000, la part base de recollida ja esta feta.

In [ ]:
print("CSV generats:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges:", len(list(images_dir.glob("*.png"))))
print("Logs:", sorted(path.name for path in logs_dir.iterdir()))